In [1]:
import json

import pprint as pp

In [2]:
model_to_use = 'llama3.2:3b'

In [3]:
text_to_tag = """AI will be in the majority of premium smartphones and wearables within a few years - bad news for anyone who doesn't like or trust the overhyped pixie dust. Counterpoint Research forecasts that more than 80 percent of premium smartphones will have agentic AI capabilities by 2027, while a similar proportion of so-called wearable devices are on track to be AI-enabled by 2032. To some degree, this appears to be a push from the vendors, who see AI as a "premium" feature to justify the inflating price tag attached to devices. Counterpoint says that MediaTek became the first chipset maker to commercialize agentic AI capabilities via its Dimensity 9400 series, followed by Qualcomm with the Snapdragon 8 Elite Gen 5 and Snapdragon 8 Gen 5 platforms. This marked the start of a new smartphone technology cycle in which devices increasingly shifted from sporting AI assistants to boasting "autonomous, context-aware AI experiences," Counterpoint claims. It defines an agentic AI smartphone as one capable of running software agents that can understand context, plan actions, make decisions, and execute multi-step tasks on behalf of the user. This places more emphasis on memory bandwidth and sustained AI throughput rather than just having a neural processing unit (NPU) to boost processing, hence the appearance of newer silicon designed with agentic AI in mind. With the memory shortage pushing up the price of phones, the device makers also need something to convince buyers to part with more of their hard-earned cash. "We expect one in three smartphones sold in 2027 to have agentic AI capability, driven by both premium (>$600) and mid-high ($250-$600) price tier smartphones," says Counterpoint research vice president Peter Richardson. However, for premium devices, the figure is 80 percent or higher, and the bigger opportunity will open up when these features start reaching mid-tier smartphones at scale, the firm forecasts. Not everyone welcomes AI in their personal gadgets. One UK used device biz reported a slump in demand for pre-owned Samsung Galaxy phones since the firm started adding AI capabilities. The figure of 80 percent crops up again in wearables, where the proportion of AI-capable devices is projected to rise from 30 percent in 2025 to nearly 80 percent by 2032. This represents a trillion-dollar revenue opportunity for the vendors, Counterpoint believes. Wearables - smartwatches, health monitors and the like - increasingly execute inference workloads locally, with models trained in the cloud then deployed onto the device. This shifts latency-sensitive functions, such as continuous health monitoring, gesture recognition, and contextual awareness to the device itself while improving privacy by cutting back on sensitive biometric information sent to the cloud, according to Counterpoint. Smartwatches and wireless earbuds are forecast to remain the largest categories by unit volume through 2032, with the latter gaining AI-driven features such as real-time language translation, speaker identification, and personalized hearing adaptation. Counterpoint expects smart rings (no giggling at the back there) to be the fastest-growing segment. This is because constantly worn items can continuously track health signals including heart rate variability, sleep stages, and stress. Revenue from AI-enabled wearables is forecast to grow at an average of 21 percent annually between now and 2032. ®"""

In [15]:
import ollama

def get_tags_as_json(text_to_tag, model_to_use = 'llama3.2:3b'):

    installed_models = [dict(x)['model'] for x in ollama.list()['models']]
    assert model_to_use in installed_models
    
    prompt_part_1 = """You are an advanced text analysis and semantic tagging system. Your task is to analyze the provided text and extract a list of rationalized, meaningful tags that accurately represent the core topics, themes, or entities in the content.

For each tag, you must also provide a brief, 1-2 sentence rationale explaining exactly why it applies based on the text.

Constraints:
1. Tags must be concise (1-3 words) and highly relevant.
2. The rationale must be grounded directly in the text (no hallucinations).
3. Do not include redundant tags or duplicates.
4. Output the final result in the exact JSON format specified below.
5. Do NOT include any conversational filler, introductions, explanations, or conclusions.
6. Do NOT use code block fences (like ```json or ```).
7. Ensure all keys and string values are enclosed in double quotes. 
8. Ensure arrays and objects are perfectly closed.
9. Exclude dates and years
10. Report all nouns as singular

Spaces between words are okay.

# Text to analyze:

"""

    prompt_part_2 = """
    
    # Desired Output Format (Strict JSON):
{
  "tags": [
    {
      "tag": "string",
      "rationale": "string"
    }
  ]
}
"""    

    prompt = prompt_part_1 + text_to_tag + prompt_part_2

    response: ollama.ChatResponse = ollama.chat(
        model = model_to_use,
        messages = [
          {
            'role': 'user',
            'content': prompt,
          },
        ]
    )
    return response.message.content

def get_tags_as_json_once(text_to_tag, model_to_use = 'llama3.2:3b'):
    worked = False
    dict_tags = None
    while not worked:
        try:
            json_tags = get_tags_as_json(text_to_tag, model_to_use = model_to_use)
            dict_tags = json.loads(json_tags)
            worked = True
        except:
            worked = False
    return dict_tags

In [16]:
dict_refined_replacement = {
    'artificial intelligence' : 'ai',
}

def multi_tag(text_to_tag, model_to_use = 'llama3.2:3b', n = 5, space_character = ' '):
    list_tags = []
    for i in range(0, n):
        dict_tags = get_tags_as_json_once(text_to_tag, model_to_use = model_to_use)
        list_tags.extend([x['tag'].strip().lower().replace('_', space_character).replace(' ', space_character) for x in dict_tags['tags']])

    list_tags = [dict_refined_replacement[x] if x in dict_refined_replacement else x for x in list_tags]
    return sorted(list(set(list_tags)))

In [17]:
multi_tag(text_to_tag, model_to_use = model_to_use)

['agentic ai',
 'ai',
 'counterpoint research',
 'economy and revenue',
 'industry outlook',
 'industry trends',
 'market opportunity',
 'market trends',
 'mediatek',
 'neural processing unit (npu)',
 'premium devices',
 'premium smartphones',
 'qualcomm',
 'revenue growth',
 'revenue opportunities',
 'smartphones',
 'technology advancements',
 'wearables']